<a href="https://colab.research.google.com/github/PreethamHD/DP-MMFL/blob/main/notebooks/07_report_normalization.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
from pathlib import Path
import pandas as pd
from google.colab import drive

drive.mount("/content/drive")

PROJECT_ROOT = Path("/content/drive/MyDrive/DP-MMFL")
MANIFEST_PATH = (
    PROJECT_ROOT
    / "data"
    / "processed"
    / "chexpert_plus_manifest.parquet"
)

manifest = pd.read_parquet(MANIFEST_PATH)
print("Shape:", manifest.shape)
print("Missing reports:", manifest["report"].isna().sum())

# Establish raw character baseline in memory if not already present
if "report_char_length" not in manifest.columns:
    manifest["report_char_length"] = manifest["report"].str.len()

Mounted at /content/drive
Shape: (223462, 54)
Missing reports: 0


In [ ]:
patterns = {
    "anonymized_disclaimer": "This report has been anonymized",
    "accession_number":      "ACCESSION NUMBER:",
    "physician_consult":     "Physician to Physician Radiology Consult Line",
    "narrative":             "NARRATIVE:",
    "impression":            "IMPRESSION:",
    "findings":              "FINDINGS:",
    "comparison":            "COMPARISON:",
}

total_rows = len(manifest)
print("--- Systematic Boilerplate & Section Header Prevalence ---")
for name, pattern in patterns.items():
    count = (
        manifest["report"]
        .str.contains(pattern, case=False, na=False, regex=False)
        .sum()
    )
    print(f"{name:25s}: {count:7d} ({count / total_rows * 100:6.2f}%)")

--- Systematic Boilerplate & Section Header Prevalence ---
anonymized_disclaimer    :  223462 (100.00%)
accession_number         :  223462 (100.00%)
physician_consult        :    9394 (  4.20%)
narrative                :  223434 ( 99.99%)
impression               :  223462 (100.00%)
findings                 :   57691 ( 25.82%)
comparison               :  211551 ( 94.67%)


In [ ]:
print("--- Sample Report Terminations (Last 800 Characters) ---")
for i, report in enumerate(manifest["report"].sample(5, random_state=42)):
    print(f"\n{'=' * 30} Sample {i + 1} {'=' * 30}")
    print(report[-800:])

--- Sample Report Terminations (Last 800 Characters) ---

============================== Sample 1 ==============================
L OF THE LEFT INTERNAL JUGULAR SHEATH AND
INTERVAL REMOVAL OF THE FEEDING TUBE. STABLE ENDOTRACHEAL TUBE AND
LEFT UPPER EXTREMITY PICC LINE.
2. INTERVAL INCREASE IN OPACIFICATION OF THE LEFT LOWER LOBE
CONSISTENT WITH WORSENING CONSOLIDATION.
3. STABLE DIFFUSE INCREASED RETICULAR OPACITIES THROUGHOUT THE
REMAINDER OF BOTH LUNGS WHICH MAY BE SECONDARY TO PULMONARY EDEMA
OR PART OF THE LEFT LOWER LOBE INFECTIOUS PROCESS.
END OF IMPRESSION
SUMMARY: 4 POSSIBLE SIGNIFICANT FINDINGS, MAY NEED ACTION
I have personally reviewed the images for this examination and agree
with the report transcribed above.
By: Kamryn Brynn Pineda, MD  on: 1-15-2015
 
ACCESSION NUMBER:
5078-8896
This report has been anonymized. All dates are offset from the actual dates by a fixed interval associated with the patient.

============================== Sample 2 =============================

In [ ]:
accession_mask = manifest["report"].str.contains(
    "ACCESSION NUMBER:",
    case=False,
    na=False,
    regex=False,
)
accession_present = manifest[accession_mask].copy()

print("Reports containing 'ACCESSION NUMBER:':", len(accession_present))

print("\nSample Accession Number Extractions:")
sample_accessions = (
    accession_present["report"]
    .str.extract(r"ACCESSION NUMBER:\s*([^\n\r]+)", expand=False)
    .head(20)
)
print(sample_accessions.to_string(index=False))

Reports containing 'ACCESSION NUMBER:': 223462

Sample Accession Number Extractions:
       LFEWOZWVDRX
        6340977630
        #624515709
           9944467
         937973082
        6340977630
        6340977630
          61738273
         #80273744
        8795694723
         #04376790
          52282814
         456646565
          hwZpMDqr
#226.435.949.823.7
       27743597930
     7154407945477
             76528
          49405268
        1366196893


In [ ]:
import re

def normalize_report(text: str) -> str:
    """
    Conservative normalization of a CheXpert Plus report.
    Standardizes line breaks and collapses irregular whitespace without
    dropping clinical sections, headers, or entities.
    """
    text = str(text)
    # Standardize newline characters
    text = text.replace("\r\n", "\n").replace("\r", "\n")
    # Collapse consecutive whitespace and newlines to a single space
    text = re.sub(r"\s+", " ", text)
    # Strip boundary whitespace
    return text.strip()

# Visual sanity verification on first instance
example = manifest["report"].iloc[0]
normalized = normalize_report(example)

print("--- Original Raw Report ---")
print(example[:350] + ("..." if len(example) > 350 else ""))

print("\n--- Conservatively Normalized Report ---")
print(normalized[:350] + ("..." if len(normalized) > 350 else ""))

--- Original Raw Report ---
NARRATIVE:
CHEST, ONE VIEW: 2-10-2001
FINDINGS: Costophrenic angles sharp, without evidence of effusion.
The cardiomediastinal silhouette is normal. Vessels mildly
indistinct with prominence of interstitial structures, suggesting
mild, pulmonary edema. Left subclavian central venous catheter is
seen, tip in mid SVC. No pneumothorax.
IMPRESSION:
1. ...

--- Conservatively Normalized Report ---
NARRATIVE: CHEST, ONE VIEW: 2-10-2001 FINDINGS: Costophrenic angles sharp, without evidence of effusion. The cardiomediastinal silhouette is normal. Vessels mildly indistinct with prominence of interstitial structures, suggesting mild, pulmonary edema. Left subclavian central venous catheter is seen, tip in mid SVC. No pneumothorax. IMPRESSION: 1. ...


In [ ]:
manifest["normalized_report"] = manifest["report"].map(normalize_report)
manifest["normalized_char_length"] = manifest["normalized_report"].str.len()

print("--- Character Length Comparison (Raw vs Normalized) ---")
print(manifest[["report_char_length", "normalized_char_length"]].describe())

quantiles = [0.50, 0.75, 0.90, 0.95, 0.99, 0.995, 1.00]
print("\n--- Quantiles Comparison ---")
print(manifest[["report_char_length", "normalized_char_length"]].quantile(quantiles))

--- Character Length Comparison (Raw vs Normalized) ---
       report_char_length  normalized_char_length
count       223462.000000           223462.000000
mean           842.414858              821.063622
std            271.541716              266.656774
min            254.000000              252.000000
25%            673.000000              654.000000
50%            789.000000              769.000000
75%            943.000000              920.000000
max           5662.000000             5504.000000

--- Quantiles Comparison ---
       report_char_length  normalized_char_length
0.500             789.000                   769.0
0.750             943.000                   920.0
0.900            1162.000                  1135.0
0.950            1347.000                  1318.0
0.990            1818.000                  1776.0
0.995            2043.695                  1999.0
1.000            5662.000                  5504.0


In [ ]:
empty_after = (manifest["normalized_report"].str.strip() == "").sum()
reports_changed = (manifest["report"] != manifest["normalized_report"]).sum()

print("Empty reports after normalization (Target: 0):", empty_after)
print(f"Total reports modified: {reports_changed:,} / {len(manifest):,}")

assert empty_after == 0, "Normalization produced blank reports!"
print("Zero-loss normalization verification passed successfully.")

Empty reports after normalization (Target: 0): 0
Total reports modified: 223,462 / 223,462
Zero-loss normalization verification passed successfully.


In [ ]:
#removing administrative records

In [ ]:
import re
import pandas as pd

def extract_after_accession(text: str) -> str:
    match = re.search(r"ACCESSION NUMBER:\s*", str(text), flags=re.IGNORECASE)
    if match:
        return str(text)[match.start():]
    return ""

accession_suffix = manifest["report"].map(extract_after_accession)
suffix_count = (accession_suffix != "").sum()
print(f"Reports with accession suffix: {suffix_count:,} / {len(manifest):,} ({suffix_count / len(manifest) * 100:.2f}%)")

print("\n--- 5 Sample Administrative Footers ---")
for i, suffix in enumerate(accession_suffix.sample(5, random_state=42)):
    print(f"\n{'=' * 30} Sample Footer {i + 1} {'=' * 30}")
    print(suffix.strip())

Reports with accession suffix: 223,462 / 223,462 (100.00%)

--- 5 Sample Administrative Footers ---

============================== Sample Footer 1 ==============================
ACCESSION NUMBER:
5078-8896
This report has been anonymized. All dates are offset from the actual dates by a fixed interval associated with the patient.

============================== Sample Footer 2 ==============================
ACCESSION NUMBER:
0210004
This report has been anonymized. All dates are offset from the actual dates by a fixed interval associated with the patient.

============================== Sample Footer 3 ==============================
ACCESSION NUMBER:
AYRAWJTQBRAG
This report has been anonymized. All dates are offset from the actual dates by a fixed interval associated with the patient.

============================== Sample Footer 4 ==============================
ACCESSION NUMBER:
5851643010
This report has been anonymized. All dates are offset from the actual dates by a fixed interval

In [ ]:
#verifying no clinical section headers exist downstream of ACCESSION NUMBER
clinical_headers = [
    "NARRATIVE:",
    "FINDINGS:",
    "IMPRESSION:",
    "SUMMARY:",
    "CLINICAL HISTORY:",
    "COMPARISON:",
    "TECHNIQUE:",
]

post_accession_clinical = {header: 0 for header in clinical_headers}

for report in manifest["report"]:
    rep_str = str(report).upper()
    accession_pos = rep_str.find("ACCESSION NUMBER:")
    if accession_pos == -1:
        continue

    after_accession = rep_str[accession_pos:]
    for header in clinical_headers:
        if header in after_accession:
            post_accession_clinical[header] += 1

print("--- Clinical Headers Found After ACCESSION NUMBER (Target: All 0) ---")
print(pd.Series(post_accession_clinical))

--- Clinical Headers Found After ACCESSION NUMBER (Target: All 0) ---
NARRATIVE:            0
FINDINGS:             0
IMPRESSION:          10
SUMMARY:              4
CLINICAL HISTORY:     3
COMPARISON:           7
TECHNIQUE:            0
dtype: int64


In [ ]:
import re

clinical_headers = [
    "NARRATIVE:",
    "FINDINGS:",
    "IMPRESSION:",
    "SUMMARY:",
    "CLINICAL HISTORY:",
    "COMPARISON:",
    "TECHNIQUE:",
]

exception_rows = []

for idx, report in manifest["report"].items():

    accession_pos = report.upper().find(
        "ACCESSION NUMBER:"
    )

    if accession_pos == -1:
        continue

    after_accession = report[
        accession_pos:
    ].upper()

    found_headers = [
        header
        for header in clinical_headers
        if header in after_accession
    ]

    if found_headers:
        exception_rows.append({
            "index": idx,
            "headers": found_headers,
            "report": report,
        })

exceptions = pd.DataFrame(exception_rows)

print(
    "Exception reports:",
    len(exceptions)
)

print(
    exceptions["headers"]
    .explode()
    .value_counts()
)

Exception reports: 10
headers
IMPRESSION:          10
COMPARISON:           7
SUMMARY:              4
CLINICAL HISTORY:     3
Name: count, dtype: int64


In [ ]:
for _, row in exceptions.iterrows():

    print("\n" + "=" * 100)
    print("INDEX:", row["index"])
    print("HEADERS FOUND:", row["headers"])
    print("=" * 100)

    print(row["report"])


INDEX: 5294
HEADERS FOUND: ['IMPRESSION:', 'SUMMARY:', 'COMPARISON:']
NARRATIVE:
CHEST ONE VIEW OF 11/18/2012 AT 1100 HOURS
CLINICAL HISTORY: KUB for tube placement.
IMPRESSION:
NO FEEDING OR NG TUBE SEEN. THE VISUALIZED BOWEL GAS PATTERN IS
NONOBSTRUCTIVE. THERE IS A FOLEY CATHETER PRESENT. SMALL
CALCIFICATIONS SEEN IN THE RIGHT LOWER PELVIS IS MOST LIKELY A
PHLEBOLITH.
END OF IMPRESSION:
CHEST ONE VIEW OF 11-18-2012, (ACCESSION NUMBER: #yHswUmOGr):
IMPRESSION:
INTERVAL PLACEMENT OF AN ET TUBE WITH THE TIP AT THE THORACIC
INLET. INTERNAL JUGULAR CATHETER HAS ITS TIP IN THE SUPERIOR VENA
CAVA. LUNG VOLUMES ARE LOW. BIBASILAR OPACITIES ARE LIKELY DUE TO
ATELECTASIS.
END OF IMPRESSION:
ABDOMEN:
COMPARISON: 11-18-2012
IMPRESSION:
ET TUBE IS IN ADEQUATE POSITION. NG TUBE IS PRESENT. LUNG VOLUMES
ARE LOW AND THERE IS IMPROVEMENT IN PULMONARY EDEMA WITH ASYMMETRIC
LEFT ALVEOLAR OPACIFICATION, WHICH MAY BE SECONDARY TO A ASYMMETRIC
PULMONARY EDEMA.
END OF IMPRESSION:
SUMMARY: 2
 
ACCESSION N

In [ ]:
#Safe cleaner implementation
import re
import pandas as pd

ANONYMIZATION_TEXT = (
    "This report has been anonymized. "
    "All dates are offset from the actual dates by a fixed interval "
    "associated with the patient."
)

def normalize_report(text: str) -> str:
    """Standardize line breaks and collapse redundant whitespace."""
    text = str(text)
    text = text.replace("\r\n", "\n").replace("\r", "\n")
    text = re.sub(r"\s+", " ", text)
    return text.strip()

def clean_report(text: str) -> str:
    """
    Targeted CheXpert Plus report cleaning.
    Removes ONLY the final administrative accession and anonymization block
    anchored at the end of the text, preserving all upstream clinical sections.
    """
    text = normalize_report(text)

    # Anchored specifically to the end ($) to avoid cutting prior accession numbers
    pattern = (
        r"\s*ACCESSION NUMBER:\s*"
        r"[^\s]+"
        r"\s*" +
        re.escape(ANONYMIZATION_TEXT) +
        r"\s*$"
    )

    cleaned = re.sub(pattern, "", text, flags=re.IGNORECASE)
    return cleaned.strip()

In [ ]:

# Tests edge cases where accession numbers appeared earlier or multiple times
if "exceptions" in globals() and isinstance(exceptions, pd.DataFrame) and len(exceptions) > 0:
    for _, row in exceptions.iterrows():
        cleaned = clean_report(row["report"])
        print("\n" + "=" * 100)
        print("INDEX:       ", row.name if "index" not in row else row["index"])
        print("RAW LENGTH:  ", len(row["report"]))
        print("CLEAN LENGTH:", len(cleaned))
        print("CLEANED REPORT:\n", cleaned)
else:
    print("No exceptions subset defined; skipping isolated check.")


INDEX:        5294
RAW LENGTH:   1067
CLEAN LENGTH: 915
CLEANED REPORT:
 NARRATIVE: CHEST ONE VIEW OF 11/18/2012 AT 1100 HOURS CLINICAL HISTORY: KUB for tube placement. IMPRESSION: NO FEEDING OR NG TUBE SEEN. THE VISUALIZED BOWEL GAS PATTERN IS NONOBSTRUCTIVE. THERE IS A FOLEY CATHETER PRESENT. SMALL CALCIFICATIONS SEEN IN THE RIGHT LOWER PELVIS IS MOST LIKELY A PHLEBOLITH. END OF IMPRESSION: CHEST ONE VIEW OF 11-18-2012, (ACCESSION NUMBER: #yHswUmOGr): IMPRESSION: INTERVAL PLACEMENT OF AN ET TUBE WITH THE TIP AT THE THORACIC INLET. INTERNAL JUGULAR CATHETER HAS ITS TIP IN THE SUPERIOR VENA CAVA. LUNG VOLUMES ARE LOW. BIBASILAR OPACITIES ARE LIKELY DUE TO ATELECTASIS. END OF IMPRESSION: ABDOMEN: COMPARISON: 11-18-2012 IMPRESSION: ET TUBE IS IN ADEQUATE POSITION. NG TUBE IS PRESENT. LUNG VOLUMES ARE LOW AND THERE IS IMPROVEMENT IN PULMONARY EDEMA WITH ASYMMETRIC LEFT ALVEOLAR OPACIFICATION, WHICH MAY BE SECONDARY TO A ASYMMETRIC PULMONARY EDEMA. END OF IMPRESSION: SUMMARY: 2

INDEX:   

In [ ]:

print("Validating footer removal across all reports...")

remaining_admin_footer = manifest["report"].apply(
    lambda x: ANONYMIZATION_TEXT.lower() in clean_report(x).lower()
)

footer_leak_count = remaining_admin_footer.sum()
print(f"Reports still containing anonymization footer (Target: 0): {footer_leak_count}")
assert footer_leak_count == 0, f"Detected {footer_leak_count} reports where the anonymization footer persisted!"

Validating footer removal across all reports...
Reports still containing anonymization footer (Target: 0): 5175


AssertionError: Detected 5175 reports where the anonymization footer persisted!

In [ ]:
# new cleaner func
import re

ANONYMIZATION_TEXT = (
    "This report has been anonymized. "
    "All dates are offset from the actual dates by a fixed interval "
    "associated with the patient."
)

def clean_report(text: str) -> str:
    text = normalize_report(text)

    # The anonymization sentence marks the end of the report.
    # Remove the sentence and the administrative accession block
    # immediately preceding it.
    pattern = (
        r"\s*ACCESSION\s+NUMBER\s*:\s*"
        r".*?"
        r"\s*" + re.escape(ANONYMIZATION_TEXT) + r"\s*$"
    )

    cleaned = re.sub(
        pattern,
        "",
        text,
        flags=re.IGNORECASE,
    )

    return cleaned.strip()

In [ ]:
for _, row in exceptions.iterrows():
    cleaned = clean_report(row["report"])

    print("\n" + "=" * 100)
    print("INDEX:", row["index"])
    print("RAW:", len(row["report"]), "chars")
    print("CLEAN:", len(cleaned), "chars")
    print("-" * 100)
    print(cleaned[-500:])


INDEX: 5294
RAW: 1067 chars
CLEAN: 354 chars
----------------------------------------------------------------------------------------------------
NARRATIVE: CHEST ONE VIEW OF 11/18/2012 AT 1100 HOURS CLINICAL HISTORY: KUB for tube placement. IMPRESSION: NO FEEDING OR NG TUBE SEEN. THE VISUALIZED BOWEL GAS PATTERN IS NONOBSTRUCTIVE. THERE IS A FOLEY CATHETER PRESENT. SMALL CALCIFICATIONS SEEN IN THE RIGHT LOWER PELVIS IS MOST LIKELY A PHLEBOLITH. END OF IMPRESSION: CHEST ONE VIEW OF 11-18-2012, (

INDEX: 25654
RAW: 794 chars
CLEAN: 75 chars
----------------------------------------------------------------------------------------------------
NARRATIVE: Patient name: NAME NAME. Medical record number: &lt;DELETED&gt;.

INDEX: 25656
RAW: 797 chars
CLEAN: 75 chars
----------------------------------------------------------------------------------------------------
NARRATIVE: Patient name: NAME NAME. Medical record number: &lt;DELETED&gt;.

INDEX: 26884
RAW: 1310 chars
CLEAN: 72 chars
--------

In [ ]:
print("Validating footer removal across all reports...")

remaining_admin_footer = manifest["report"].apply(
    lambda x: ANONYMIZATION_TEXT.lower() in clean_report(x).lower()
)

footer_leak_count = remaining_admin_footer.sum()

print(
    f"Reports still containing anonymization footer "
    f"(Target: 0): {footer_leak_count}"
)

Validating footer removal across all reports...
Reports still containing anonymization footer (Target: 0): 0


In [ ]:
print("\nChecking clinical headers before vs after cleaning...")

for header in clinical_headers:
    before = manifest["report"].str.upper().str.contains(
        header,
        regex=False
    ).sum()

    after = manifest["report"].apply(
        lambda x: header in clean_report(x).upper()
    ).sum()

    print(
        f"{header:20s} "
        f"before={before:>7,} "
        f"after={after:>7,} "
        f"removed={before-after:>7,}"
    )


Checking clinical headers before vs after cleaning...
NARRATIVE:           before=223,434 after=223,434 removed=      0
FINDINGS:            before= 57,691 after= 57,691 removed=      0
IMPRESSION:          before=223,462 after=223,457 removed=      5
SUMMARY:             before=163,174 after=163,170 removed=      4
CLINICAL HISTORY:    before=138,893 after=138,894 removed=     -1
COMPARISON:          before=211,551 after=211,546 removed=      5
TECHNIQUE:           before= 10,689 after= 10,689 removed=      0


In [ ]:
# Find reports where clinical headers disappeared after cleaning

header_changes = []

for idx, report in manifest["report"].items():
    cleaned = clean_report(report)

    for header in clinical_headers:
        before = header in report.upper()
        after = header in cleaned.upper()

        if before and not after:
            header_changes.append({
                "index": idx,
                "header": header,
                "raw": report,
                "cleaned": cleaned,
            })

header_changes_df = pd.DataFrame(header_changes)

print("Total header removals:", len(header_changes_df))

print("\nRemovals by header:")
print(header_changes_df["header"].value_counts())

print("\nAffected indices:")
print(header_changes_df[["index", "header"]].to_string(index=False))

Total header removals: 17

Removals by header:
header
COMPARISON:          5
IMPRESSION:          5
SUMMARY:             4
CLINICAL HISTORY:    3
Name: count, dtype: int64

Affected indices:
 index            header
  5294          SUMMARY:
  5294       COMPARISON:
 25654       IMPRESSION:
 25654          SUMMARY:
 25654 CLINICAL HISTORY:
 25654       COMPARISON:
 25656       IMPRESSION:
 25656          SUMMARY:
 25656 CLINICAL HISTORY:
 25656       COMPARISON:
 26884       IMPRESSION:
 26884       COMPARISON:
 32519       IMPRESSION:
112623       IMPRESSION:
112623          SUMMARY:
112623 CLINICAL HISTORY:
112623       COMPARISON:


In [ ]:
for _, row in header_changes_df.iterrows():
    print("\n" + "=" * 100)
    print("INDEX:", row["index"])
    print("HEADER REMOVED:", row["header"])
    print("-" * 100)

    print("RAW:")
    print(row["raw"])

    print("\nCLEANED:")
    print(row["cleaned"])


INDEX: 5294
HEADER REMOVED: SUMMARY:
----------------------------------------------------------------------------------------------------
RAW:
NARRATIVE:
CHEST ONE VIEW OF 11/18/2012 AT 1100 HOURS
CLINICAL HISTORY: KUB for tube placement.
IMPRESSION:
NO FEEDING OR NG TUBE SEEN. THE VISUALIZED BOWEL GAS PATTERN IS
NONOBSTRUCTIVE. THERE IS A FOLEY CATHETER PRESENT. SMALL
CALCIFICATIONS SEEN IN THE RIGHT LOWER PELVIS IS MOST LIKELY A
PHLEBOLITH.
END OF IMPRESSION:
CHEST ONE VIEW OF 11-18-2012, (ACCESSION NUMBER: #yHswUmOGr):
IMPRESSION:
INTERVAL PLACEMENT OF AN ET TUBE WITH THE TIP AT THE THORACIC
INLET. INTERNAL JUGULAR CATHETER HAS ITS TIP IN THE SUPERIOR VENA
CAVA. LUNG VOLUMES ARE LOW. BIBASILAR OPACITIES ARE LIKELY DUE TO
ATELECTASIS.
END OF IMPRESSION:
ABDOMEN:
COMPARISON: 11-18-2012
IMPRESSION:
ET TUBE IS IN ADEQUATE POSITION. NG TUBE IS PRESENT. LUNG VOLUMES
ARE LOW AND THERE IS IMPROVEMENT IN PULMONARY EDEMA WITH ASYMMETRIC
LEFT ALVEOLAR OPACIFICATION, WHICH MAY BE SECONDARY TO 

In [ ]:
# pakka this is last cleaner T_T
import re

ANONYMIZATION_TEXT = (
    "This report has been anonymized. "
    "All dates are offset from the actual dates by a fixed interval "
    "associated with the patient."
)

def clean_report(text: str) -> str:
    text = normalize_report(text)

    # The final footer has:
    # ACCESSION NUMBER:
    # <single accession value>
    # This report has been anonymized...
    #
    # Since normalize_report() has already collapsed whitespace,
    # require the accession value to be a single non-whitespace token.
    pattern = (
        r"\s+ACCESSION\s+NUMBER\s*:\s*"
        r"[^\s]+"
        r"\s+" +
        re.escape(ANONYMIZATION_TEXT) +
        r"\s*$"
    )

    return re.sub(
        pattern,
        "",
        text,
        flags=re.IGNORECASE,
    ).strip()

In [ ]:
print("Testing corrected cleaner on exception reports...\n")

for _, row in exceptions.iterrows():
    cleaned = clean_report(row["report"])

    print("=" * 100)
    print("INDEX:", row["index"])
    print("RAW LENGTH:", len(row["report"]))
    print("CLEAN LENGTH:", len(cleaned))
    print("CLEANED END:")
    print(cleaned[-300:])

Testing corrected cleaner on exception reports...

INDEX: 5294
RAW LENGTH: 1067
CLEAN LENGTH: 915
CLEANED END:
ESSION: ABDOMEN: COMPARISON: 11-18-2012 IMPRESSION: ET TUBE IS IN ADEQUATE POSITION. NG TUBE IS PRESENT. LUNG VOLUMES ARE LOW AND THERE IS IMPROVEMENT IN PULMONARY EDEMA WITH ASYMMETRIC LEFT ALVEOLAR OPACIFICATION, WHICH MAY BE SECONDARY TO A ASYMMETRIC PULMONARY EDEMA. END OF IMPRESSION: SUMMARY: 2
INDEX: 25654
RAW LENGTH: 794
CLEAN LENGTH: 617
CLEANED END:
ARDIOPULMONARY ABNORMALITY OR PNEUMOTHORAX. 2. VISUALIZED BONY STRUCTURES WITHIN NORMAL LIMITS. 3. NORMAL HEART SIZE; UNCHANGED TORTUOSITY OF THORACIC AORTA. SUMMARY: 1-NO SIGNIFICANT ABNORMALITY I have personally reviewed the images for this examination and agreed with the report transcribed above.
INDEX: 25656
RAW LENGTH: 797
CLEAN LENGTH: 617
CLEANED END:
ARDIOPULMONARY ABNORMALITY OR PNEUMOTHORAX. 2. VISUALIZED BONY STRUCTURES WITHIN NORMAL LIMITS. 3. NORMAL HEART SIZE; UNCHANGED TORTUOSITY OF THORACIC AORTA. SUMMARY: 

In [ ]:


remaining_admin_footer = manifest["report"].apply(
    lambda x: ANONYMIZATION_TEXT.lower() in clean_report(x).lower()
)

footer_leak_count = remaining_admin_footer.sum()

print(
    "Reports still containing anonymization footer:",
    footer_leak_count
)

Reports still containing anonymization footer: 5175


In [ ]:
footer_phrase = (
    "This report has been anonymized. "
    "All dates are offset from the actual dates by a fixed interval "
    "associated with the patient."
)

leaking = []

for idx, report in manifest["report"].items():
    cleaned = clean_report(report)

    if footer_phrase.lower() in cleaned.lower():
        leaking.append({
            "index": idx,
            "raw": report,
            "cleaned": cleaned,
        })

leaking_df = pd.DataFrame(leaking)

print("Leaking reports:", len(leaking_df))

Leaking reports: 5175


In [ ]:
for _, row in leaking_df.head(10).iterrows():
    print("\n" + "=" * 100)
    print("INDEX:", row["index"])
    print("-" * 100)

    print("LAST 500 RAW CHARACTERS:")
    print(repr(row["raw"][-500:]))

    print("\nLAST 500 CLEANED CHARACTERS:")
    print(repr(row["cleaned"][-500:]))


INDEX: 55
----------------------------------------------------------------------------------------------------
LAST 500 RAW CHARACTERS:
'AL SILHOUETTE IS \nUNREMARKABLE.  THERE IS NO MEDIASTINAL OR HILAR LYMPHADENOPATHY.  \nLUNGS ARE CLEAR.  THERE IS NO PLEURAL EFFUSION OR PNEUMOTHORAX.  \nVISUALIZED OSSEOUS STRUCTURES ARE UNREMARKABLE.  \n \n SUMMARY:1-NO SIGNIFICANT ABNORMALITY  \nI have personally reviewed the images for this examination and agreed\nwith the report transcribed above.\n \nACCESSION NUMBER:\n88 16 1\nThis report has been anonymized. All dates are offset from the actual dates by a fixed interval associated with the patient.'

LAST 500 CLEANED CHARACTERS:
' CARDIOMEDIASTINAL SILHOUETTE IS UNREMARKABLE. THERE IS NO MEDIASTINAL OR HILAR LYMPHADENOPATHY. LUNGS ARE CLEAR. THERE IS NO PLEURAL EFFUSION OR PNEUMOTHORAX. VISUALIZED OSSEOUS STRUCTURES ARE UNREMARKABLE. SUMMARY:1-NO SIGNIFICANT ABNORMALITY I have personally reviewed the images for this examination and agreed wit

In [ ]:
for _, row in leaking_df.head(10).iterrows():
    report = row["raw"]

    pos = report.lower().rfind("this report has been anonymized")

    print("\n" + "=" * 100)
    print("INDEX:", row["index"])
    print("TEXT BEFORE FOOTER:")
    print(repr(report[max(0, pos - 250):]))


INDEX: 55
TEXT BEFORE FOOTER:
'USION OR PNEUMOTHORAX.  \nVISUALIZED OSSEOUS STRUCTURES ARE UNREMARKABLE.  \n \n SUMMARY:1-NO SIGNIFICANT ABNORMALITY  \nI have personally reviewed the images for this examination and agreed\nwith the report transcribed above.\n \nACCESSION NUMBER:\n88 16 1\nThis report has been anonymized. All dates are offset from the actual dates by a fixed interval associated with the patient.'

INDEX: 59
TEXT BEFORE FOOTER:
'OF IMPRESSION:\nSUMMARY: 4 POSSIBLE SIGNIFICANT FINDINGS, MAY NEED ACTION\nI have personally reviewed the images for this examination and agree\nwith the report transcribed above.\nBy: Kaydence, Reed  on: 6/4/2016\n \nACCESSION NUMBER:\n63 35 16 41 29 89 8\nThis report has been anonymized. All dates are offset from the actual dates by a fixed interval associated with the patient.'

INDEX: 69
TEXT BEFORE FOOTER:
'SIGNIFICANT ABNORMALITY/CHANGES, MAY NEED\nACTION.\nI have personally reviewed the images for this examination and agree\nwith the repo

In [ ]:
import re

ANONYMIZATION_TEXT = (
    "This report has been anonymized. "
    "All dates are offset from the actual dates by a fixed interval "
    "associated with the patient."
)

def clean_report(text: str) -> str:
    text = normalize_report(text)

    # Remove only the FINAL administrative footer.
    # The accession value may contain spaces and multiple tokens.
    pattern = (
        r"\s+ACCESSION\s+NUMBER\s*:\s*"
        r".*?"
        r"\s+" + re.escape(ANONYMIZATION_TEXT) + r"\s*$"
    )

    return re.sub(
        pattern,
        "",
        text,
        count=1,
        flags=re.IGNORECASE,
    ).strip()

In [ ]:
remaining_admin_footer = manifest["report"].apply(
    lambda x: ANONYMIZATION_TEXT.lower() in clean_report(x).lower()
)

footer_leak_count = remaining_admin_footer.sum()

print(
    f"Reports still containing anonymization footer: "
    f"{footer_leak_count}"
)

Reports still containing anonymization footer: 0


In [ ]:
header_changes = []

for idx, report in manifest["report"].items():
    cleaned = clean_report(report)

    for header in clinical_headers:
        before = header in report.upper()
        after = header in cleaned.upper()

        if before and not after:
            header_changes.append({
                "index": idx,
                "header": header,
            })

header_changes_df = pd.DataFrame(header_changes)

print("Total clinical-header removals:", len(header_changes_df))

if len(header_changes_df) > 0:
    print(header_changes_df.to_string(index=False))

Total clinical-header removals: 13
 index            header
 25654       IMPRESSION:
 25654          SUMMARY:
 25654 CLINICAL HISTORY:
 25654       COMPARISON:
 25656       IMPRESSION:
 25656          SUMMARY:
 25656 CLINICAL HISTORY:
 25656       COMPARISON:
 32519       IMPRESSION:
112623       IMPRESSION:
112623          SUMMARY:
112623 CLINICAL HISTORY:
112623       COMPARISON:


In [ ]:
for _, row in exceptions.iterrows():
    cleaned = clean_report(row["report"])

    print("\n" + "=" * 100)
    print("INDEX:", row["index"])
    print("CLEAN LENGTH:", len(cleaned))
    print("CLEANED END:")
    print(cleaned[-500:])


INDEX: 5294
CLEAN LENGTH: 915
CLEANED END:
 OF AN ET TUBE WITH THE TIP AT THE THORACIC INLET. INTERNAL JUGULAR CATHETER HAS ITS TIP IN THE SUPERIOR VENA CAVA. LUNG VOLUMES ARE LOW. BIBASILAR OPACITIES ARE LIKELY DUE TO ATELECTASIS. END OF IMPRESSION: ABDOMEN: COMPARISON: 11-18-2012 IMPRESSION: ET TUBE IS IN ADEQUATE POSITION. NG TUBE IS PRESENT. LUNG VOLUMES ARE LOW AND THERE IS IMPROVEMENT IN PULMONARY EDEMA WITH ASYMMETRIC LEFT ALVEOLAR OPACIFICATION, WHICH MAY BE SECONDARY TO A ASYMMETRIC PULMONARY EDEMA. END OF IMPRESSION: SUMMARY: 2

INDEX: 25654
CLEAN LENGTH: 75
CLEANED END:
NARRATIVE: Patient name: NAME NAME. Medical record number: &lt;DELETED&gt;.

INDEX: 25656
CLEAN LENGTH: 75
CLEANED END:
NARRATIVE: Patient name: NAME NAME. Medical record number: &lt;DELETED&gt;.

INDEX: 26884
CLEAN LENGTH: 1156
CLEANED END:
GAS-FILLED BOWEL LOOPS NOTED IN THE RIGHT ABDOMEN AND GAS WAS NOTED IN THE RECTUM WITH NO DEFINITE EVIDENCE OF OBSTRUCTION. 3. MOTION ARTIFACT WITH DIFFUSE BLURRING MAKE

In [ ]:
#finallllllll clean func
def clean_report(text: str) -> str:
    text = normalize_report(text)

    # Find the final anonymization sentence.
    anon_pos = text.lower().rfind(
        ANONYMIZATION_TEXT.lower()
    )

    if anon_pos == -1:
        return text

    # Everything after the anonymization sentence is also administrative.
    end_pos = anon_pos + len(ANONYMIZATION_TEXT)

    # Look backwards from the anonymization sentence for the
    # immediately preceding "ACCESSION NUMBER:" marker.
    prefix = text[:anon_pos]

    accession_pos = prefix.lower().rfind("accession number:")

    if accession_pos == -1:
        # No identifiable accession footer.
        return text

    # Only remove the accession block if it occurs near the end.
    # There should be no substantial clinical content between the
    # final accession marker and the anonymization sentence.
    footer_start = accession_pos

    return text[:footer_start].strip()

In [ ]:
for _, row in leaking_df.head(10).iterrows():
    cleaned = clean_report(row["raw"])

    print("\n" + "=" * 100)
    print("INDEX:", row["index"])
    print("RAW:", len(row["raw"]))
    print("CLEAN:", len(cleaned))
    print("-" * 100)
    print(cleaned[-500:])


INDEX: 55
RAW: 773
CLEAN: 581
----------------------------------------------------------------------------------------------------
 at 0938 hours. CLINICAL HISTORY: A 17-year-old male with history of positive PPD. Patient has fever and cough. IMPRESSION: CENTRAL AIRWAYS ARE PATENT. CARDIOMEDIASTINAL SILHOUETTE IS UNREMARKABLE. THERE IS NO MEDIASTINAL OR HILAR LYMPHADENOPATHY. LUNGS ARE CLEAR. THERE IS NO PLEURAL EFFUSION OR PNEUMOTHORAX. VISUALIZED OSSEOUS STRUCTURES ARE UNREMARKABLE. SUMMARY:1-NO SIGNIFICANT ABNORMALITY I have personally reviewed the images for this examination and agreed with the report transcribed above.

INDEX: 59
RAW: 736
CLEAN: 570
----------------------------------------------------------------------------------------------------
 HOURS: CLINICAL HISTORY: Aortic dissection, evaluate for infiltrates. IMPRESSION: 1. A LEFT UPPER EXTREMITY PICC LINE IS UNCHANGED IN POSITION. 2. THERE ARE LOW LUNG VOLUMES, THE LUNGS ARE OTHERWISE CLEAR WITH NO EVIDENCE OF FOCAL OPA

In [ ]:
print("Validating footer removal...")

footer_leak_count = manifest["report"].apply(
    lambda x: ANONYMIZATION_TEXT.lower() in clean_report(x).lower()
).sum()

print(
    "Reports still containing anonymization footer:",
    footer_leak_count
)

Validating footer removal...
Reports still containing anonymization footer: 0


In [ ]:
print("\nChecking that known clinical sections are preserved...")

for header in clinical_headers:
    before = manifest["report"].str.upper().str.contains(
        header,
        regex=False
    ).sum()

    after = manifest["report"].apply(
        lambda x: header in clean_report(x).upper()
    ).sum()

    print(
        f"{header:20s} "
        f"before={before:,} "
        f"after={after:,} "
        f"difference={after-before:+,}"
    )


Checking that known clinical sections are preserved...
NARRATIVE:           before=223,434 after=223,434 difference=+0
FINDINGS:            before=57,691 after=57,691 difference=+0
IMPRESSION:          before=223,462 after=223,462 difference=+0
SUMMARY:             before=163,174 after=163,174 difference=+0
CLINICAL HISTORY:    before=138,893 after=138,897 difference=+4
COMPARISON:          before=211,551 after=211,551 difference=+0
TECHNIQUE:           before=10,689 after=10,689 difference=+0


In [ ]:
#### helllllll yeahhh the final version
import re
from pathlib import Path
import pandas as pd

# 1. Canonical normalization and cleaning implementation
ANONYMIZATION_TEXT = (
    "This report has been anonymized. "
    "All dates are offset from the actual dates by a fixed interval "
    "associated with the patient."
)

def normalize_report(text: str) -> str:
    """Standardize line breaks and collapse whitespace."""
    text = str(text)
    text = text.replace("\r\n", "\n").replace("\r", "\n")
    text = re.sub(r"\s+", " ", text)
    return text.strip()

def clean_report(text: str) -> str:
    """
    Locked canonical report cleaner.
    Identifies the final anonymization disclaimer and strips
    from the immediately preceding accession number onward.
    """
    text = normalize_report(text)

    anon_pos = text.lower().rfind(ANONYMIZATION_TEXT.lower())
    if anon_pos == -1:
        return text

    prefix = text[:anon_pos]
    accession_pos = prefix.lower().rfind("accession number:")
    if accession_pos == -1:
        return text

    return text[:accession_pos].strip()

# 2. Compute/Overwrite report_clean across the entire manifest
print("Cleaning reports across full manifest...")
manifest["report_clean"] = manifest["report"].apply(clean_report)

print("Manifest shape:          ", manifest.shape)
print("Missing cleaned reports: ", manifest["report_clean"].isna().sum())
print("Empty cleaned reports:   ", (manifest["report_clean"].str.len() == 0).sum())

assert manifest["report_clean"].isna().sum() == 0, "Missing values found in report_clean!"
assert (manifest["report_clean"].str.len() == 0).sum() == 0, "Empty strings found in report_clean!"

# 3. Visual sample verification
print("\n--- Sample Comparison (First 3 Reports) ---")
for i, (_, row) in enumerate(manifest[["report", "report_clean"]].head(3).iterrows()):
    print(f"\n[Sample {i + 1}]")
    print("Raw Tail:    ", repr(row["report"][-120:]))
    print("Cleaned Tail:", repr(row["report_clean"][-120:]))

# 4. Save canonical parquet artifact
OUTPUT_PATH = Path("/content/drive/MyDrive/DP-MMFL/data/processed/chexpert_plus_manifest.parquet")
manifest.to_parquet(OUTPUT_PATH, index=False)
print(f"\nSuccessfully saved updated manifest to: {OUTPUT_PATH}")

# 5. Reload verification
reloaded = pd.read_parquet(OUTPUT_PATH, columns=["sample_id", "report", "report_clean"])
assert reloaded.shape[0] == 223462, f"Row count mismatch: {reloaded.shape[0]}"
assert not reloaded["report_clean"].isna().any(), "Reload verification failed: null values present."
print("Reload verification passed: 223,462 records confirmed.")

Cleaning reports across full manifest...
Manifest shape:           (223462, 58)
Missing cleaned reports:  0
Empty cleaned reports:    0

--- Sample Comparison (First 3 Reports) ---

[Sample 1]
Raw Tail:     ' report has been anonymized. All dates are offset from the actual dates by a fixed interval associated with the patient.'
Cleaned Tail: 'on and agree with the report transcribed above. By: Dr. Juarez Tucker N on: 2-10-2001 __________________________________'

[Sample 2]
Raw Tail:     ' report has been anonymized. All dates are offset from the actual dates by a fixed interval associated with the patient.'
Cleaned Tail: 'T THE THORACIC SPINE, WITHOUT SIGNIFICANT VERTEBRAL BODY COLLAPSE. END OF IMPRESSION: __________________________________'

[Sample 3]
Raw Tail:     ' report has been anonymized. All dates are offset from the actual dates by a fixed interval associated with the patient.'
Cleaned Tail: 'T THE THORACIC SPINE, WITHOUT SIGNIFICANT VERTEBRAL BODY COLLAPSE. END OF IMPRESSIO

In [ ]:

from transformers import AutoTokenizer

MODEL_NAME = "emilyalsentzer/Bio_ClinicalBERT"
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

print("Tokenizer:", MODEL_NAME)
print("Vocab size:", tokenizer.vocab_size)

config.json:   0%|          | 0.00/385 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/213k [00:00<?, ?B/s]

Tokenizer: emilyalsentzer/Bio_ClinicalBERT
Vocab size: 28996


In [ ]:
# Cell 2: Batch Tokenize Cleaned Reports Without Truncation
import numpy as np
from tqdm.auto import tqdm

texts = manifest["report_clean"].tolist()
token_lengths = []
BATCH_SIZE = 1000

for i in tqdm(
    range(0, len(texts), BATCH_SIZE),
    desc="Calculating token lengths"
):
    batch = texts[i:i + BATCH_SIZE]

    encoded = tokenizer(
        batch,
        add_special_tokens=True,
        truncation=False,
        padding=False,
    )

    token_lengths.extend(
        len(ids) for ids in encoded["input_ids"]
    )

token_lengths = np.asarray(token_lengths)

print("\n--- Summary Statistics ---")
print("Reports:", len(token_lengths))
print("Min:    ", token_lengths.min())
print("Max:    ", token_lengths.max())
print("Mean:   ", round(float(token_lengths.mean()), 2))
print("Median: ", float(np.median(token_lengths)))

Calculating token lengths:   0%|          | 0/224 [00:00<?, ?it/s]


--- Summary Statistics ---
Reports: 223462
Min:     23
Max:     1361
Mean:    161.66
Median:  149.0


In [ ]:
# Cell 3: Quantile Distribution of Cleaned Tokens
quantiles = [50, 75, 90, 95, 97.5, 99, 99.5, 99.9]

print("\n--- Token Length Distribution ---")
for q in quantiles:
    print(f"{q:>5}% : {np.percentile(token_lengths, q):.2f}")


--- Token Length Distribution ---
   50% : 149.00
   75% : 185.00
   90% : 236.00
   95% : 279.00
 97.5% : 325.00
   99% : 389.00
 99.5% : 443.00
 99.9% : 576.54


In [ ]:
# Cell 4: Truncation Analysis Across Candidate Limits
candidate_lengths = [128, 160, 192, 256, 320, 384, 512]

print("\n--- Truncation Analysis ---")
for max_length in candidate_lengths:
    truncated = (token_lengths > max_length).sum()
    percentage = truncated / len(token_lengths) * 100

    print(
        f"{max_length:>3} tokens : "
        f"{truncated:>7,} reports truncated "
        f"({percentage:5.2f}%)"
    )


--- Truncation Analysis ---
128 tokens : 154,652 reports truncated (69.21%)
160 tokens :  89,828 reports truncated (40.20%)
192 tokens :  48,285 reports truncated (21.61%)
256 tokens :  16,012 reports truncated ( 7.17%)
320 tokens :   6,003 reports truncated ( 2.69%)
384 tokens :   2,372 reports truncated ( 1.06%)
512 tokens :     453 reports truncated ( 0.20%)


In [2]:
%%writefile /content/DP-MMFL/src/dp_mmfl/data/text.py
from typing import List, Union
from transformers import AutoTokenizer
import torch

MODEL_NAME = "emilyalsentzer/Bio_ClinicalBERT"
MAX_LENGTH = 384

class ClinicalTextTokenizer:
    """
    Standardized tokenizer wrapper for clinical reports using Bio_ClinicalBERT.
    Enforces unified sequence truncation and padding to MAX_LENGTH (384).
    """
    def __init__(
        self,
        model_name: str = MODEL_NAME,
        max_length: int = MAX_LENGTH,
    ):
        self.model_name = model_name
        self.max_length = max_length
        self.tokenizer = AutoTokenizer.from_pretrained(model_name)

    def encode(self, text: str):
        """Encode a single report into PyTorch tensors."""
        return self.tokenizer(
            text,
            add_special_tokens=True,
            max_length=self.max_length,
            truncation=True,
            padding="max_length",
            return_attention_mask=True,
            return_tensors="pt",
        )

    def encode_batch(self, texts: Union[List[str], tuple]):
        """Encode a collection of reports into batch PyTorch tensors."""
        return self.tokenizer(
            list(texts),
            add_special_tokens=True,
            max_length=self.max_length,
            truncation=True,
            padding="max_length",
            return_attention_mask=True,
            return_tensors="pt",
        )

Writing /content/DP-MMFL/src/dp_mmfl/data/text.py


In [3]:
%cd /content/DP-MMFL
!git add src/dp_mmfl/data/text.py
!git commit -m "implemented ClinicalTextTokenizer module with MAX_LENGTH=384"
!git push origin main
!git status

/content/DP-MMFL
[main 69f6280] implemented ClinicalTextTokenizer module with MAX_LENGTH=384
 1 file changed, 44 insertions(+)
 create mode 100644 src/dp_mmfl/data/text.py
Enumerating objects: 10, done.
Counting objects: 100% (10/10), done.
Delta compression using up to 2 threads
Compressing objects: 100% (5/5), done.
Writing objects: 100% (6/6), 977 bytes | 325.00 KiB/s, done.
Total 6 (delta 3), reused 0 (delta 0), pack-reused 0
remote: Resolving deltas: 100% (3/3), completed with 3 local objects.
To https://github.com/PreethamHD/DP-MMFL.git
   b235479..69f6280  main -> main
On branch main
Your branch is up to date with 'origin/main'.

nothing to commit, working tree clean
